# 03 — SU(2) Rotations and Geometry

Every single-qubit gate is a rotation on the Bloch sphere.  The mathematical group governing these rotations is **SU(2)** — the special unitary group of 2×2 matrices.

---

## SU(2) matrices

An SU(2) matrix is a 2×2 complex unitary matrix with determinant 1:

$$U = \begin{pmatrix} \alpha & -\beta^* \\ \beta & \alpha^* \end{pmatrix}, \quad |\alpha|^2 + |\beta|^2 = 1$$

Every such matrix represents a rotation on the Bloch sphere by some angle $\theta$ around some axis $\hat{n}$.

---

## The rotation formula

$$R_{\hat{n}}(\theta) = e^{-i\theta\hat{n}\cdot\vec{\sigma}/2} = \cos\!\frac{\theta}{2}\,I - i\sin\!\frac{\theta}{2}\,(n_x X + n_y Y + n_z Z)$$

where $X, Y, Z$ are the Pauli matrices.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from helpers.notebook_style import setup_notebook
from helpers.plotting import draw_bloch_sphere, plot_rotation_path
from helpers.display_utils import show_matrix
setup_notebook()

# Pauli matrices
I  = np.eye(2, dtype=complex)
X  = np.array([[0, 1], [1, 0]], dtype=complex)
Y  = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z  = np.array([[1, 0], [0, -1]], dtype=complex)

def su2_rotation(theta, axis):
    """SU(2) rotation matrix for angle theta around unit axis (nx,ny,nz).
    NOTE: In production use rqm-core.SU2.rotation().
    """
    n = np.array(axis, dtype=float)
    n = n / np.linalg.norm(n)
    return (np.cos(theta/2) * I
            - 1j * np.sin(theta/2) * (n[0]*X + n[1]*Y + n[2]*Z))

# Show the standard gates as SU(2) rotations
print("Hadamard H = R_x+z(π):")
H = su2_rotation(np.pi, [1/np.sqrt(2), 0, 1/np.sqrt(2)])
print(np.round(H, 4))
print()
print("S gate = R_z(π/2):")
S = su2_rotation(np.pi/2, [0, 0, 1])
print(np.round(S, 4))


## Visualising a Rz rotation

Let's see how $R_z(\theta)$ rotates a state that starts at $|+\rangle$ (the equator) around the Z axis.


In [ ]:
def apply_gate(U, spinor):
    return U @ spinor

def spinor_to_bloch(spinor):
    a, b = spinor[0], spinor[1]
    return np.array([
        2 * (a.conjugate() * b).real,
        2 * (a.conjugate() * b).imag,
        abs(a)**2 - abs(b)**2,
    ])

# Start at |+> = (1,1)/sqrt(2)
psi0 = np.array([1, 1], dtype=complex) / np.sqrt(2)

thetas = np.linspace(0, 2 * np.pi, 100)
path = []
for t in thetas:
    Rz = su2_rotation(t, [0, 0, 1])
    psi = apply_gate(Rz, psi0)
    path.append(spinor_to_bloch(psi))
path = np.array(path)

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection="3d")
plot_rotation_path(
    path, ax=ax,
    title=r"$R_z(	heta)$ applied to $|+angle$",
)
plt.tight_layout()
plt.show()


## Key gates as rotations

| Gate | Rotation | Axis | Angle |
|---|---|---|---|
| $X$ | $R_x(\pi)$ | X | 180° |
| $Y$ | $R_y(\pi)$ | Y | 180° |
| $Z$ | $R_z(\pi)$ | Z | 180° |
| $H$ | $R_{x+z}(\pi)$ | X+Z | 180° |
| $S$ | $R_z(\pi/2)$ | Z | 90° |
| $T$ | $R_z(\pi/4)$ | Z | 45° |

This geometric view makes it clear why $HZH = X$: reflecting through the X+Z diagonal then the Z axis is the same as reflecting through the X axis.


In [ ]:
# Verify HZH = X
H_gate = su2_rotation(np.pi, [1/np.sqrt(2), 0, 1/np.sqrt(2)])
Z_gate = su2_rotation(np.pi, [0, 0, 1])
HZH = H_gate @ Z_gate @ H_gate

print("HZH =")
print(np.round(HZH, 4))
print()
print("X =")
print(np.round(X, 4))
print()
# Check equality up to global phase
ratio = HZH / X
print(f"Global phase factor: {ratio[0,0]:.4f}  (should be ±1)")


## Summary

- Every single-qubit gate is an SU(2) rotation on the Bloch sphere.
- The rotation angle is $\theta$ and the axis is $\hat{n}$.
- Gate composition = rotation composition = quaternion multiplication.
- `rqm-core` provides `SU2.rotation(theta, axis)` for all of this.

**Next:** [04_rqm_core_as_source_of_truth.ipynb](04_rqm_core_as_source_of_truth.ipynb)
